# Guided Soft Actor-Critic: experiments

Reference: Haklidir, M. and Temeltas, H. (2021). Guided Soft Actor Critic: A Guided Deep Reinforcement Learning Approach for Partially Observable Markov Decision Processes. IEEE Access, 9, 159672-159683.

This notebook runs the Guided SAC implementation on five MuJoCo POMDP tasks and compares it with the SAC baseline on partial observations. Sections: setup, the Guided SAC module, the baseline, a short validation run, the full experiments under the paper's protocol (1M steps, 10 seeds), results and learning curves (Table 3 and Figure 5), and export.

## 1. Setup

In [ ]:
%%capture
!pip install mujoco gymnasium torch numpy matplotlib pandas tqdm

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
import time
from collections import deque
from copy import deepcopy
from tqdm.auto import tqdm

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")

# Verify MuJoCo
try:
    env = gym.make("Hopper-v4")
    obs, _ = env.reset()
    print(f"MuJoCo OK, Hopper-v4 obs shape: {obs.shape}")
    env.close()
except Exception as e:
    print(f"MuJoCo ERROR: {e}")

## 2. Guided SAC

The cells below hold the components of the paper: the iterative interaction loop (Eq. 11, Figure 2), the observation history (K = 4), the DNN distillation module (Figure 2, Step 9), the twin Q-network with consistent updates (Eq. 12 to 17) and automatic entropy tuning.

In [ ]:
# Configuration

INACTIVE_DIMS = {
    "HalfCheetah-v4":            [4, 5, 9],
    "Hopper-v4":                 [3, 4, 5, 6],
    "InvertedDoublePendulum-v4": [0, 3],
    "Walker2d-v4":               [8, 9, 13],
    "Swimmer-v4":                [1, 2, 3, 4, 5],
}

DEFAULT_CONFIG = {
    "gamma":            0.99,
    "tau":              0.005,
    "lr":               3e-4,
    "alpha_lr":         3e-4,
    "batch_size":       256,
    "replay_capacity":  int(1e6),
    "hidden_dim":       256,
    "total_steps":      int(1e6),
    "random_start":     10000,
    "update_after":     1000,
    "update_every":     1,
    "eval_every":       10000,
    "eval_episodes":    10,
    "history_len":      4,
    "init_alpha":       0.2,
    "distill_lr":       3e-4,
    "distill_coef":     0.1,
    "distill_update_every": 1,
}

def get_partial_obs(obs, env_name):
    partial = obs.copy()
    for idx in INACTIVE_DIMS.get(env_name, []):
        if idx < len(partial):
            partial[idx] = 0.0
    return partial

def soft_update(target, source, tau):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1.0 - tau) * tp.data)

def to_tensor(x, device=DEVICE):
    return torch.as_tensor(x, dtype=torch.float32, device=device)

In [ ]:
# Observation History (Paper Section V.B: K=4)

class ObservationHistory:
    def __init__(self, obs_dim, history_len=4):
        self.obs_dim = obs_dim
        self.history_len = history_len
        self.history = deque(maxlen=history_len + 1)

    def reset(self, obs):
        self.history.clear()
        for _ in range(self.history_len + 1):
            self.history.append(obs.copy())

    def push(self, obs):
        self.history.append(obs.copy())

    def get(self):
        return np.concatenate(list(self.history), axis=0)

    @property
    def total_dim(self):
        return self.obs_dim * (self.history_len + 1)

In [ ]:
# Neural Networks

class GaussianPolicy(nn.Module):
    LOG_STD_MIN, LOG_STD_MAX = -20, 2

    def __init__(self, input_dim, action_dim, hidden_dim=256, action_scale=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
        self.action_scale = action_scale

    def forward(self, x):
        h = self.net(x)
        mean = self.mean_head(h)
        log_std = torch.clamp(self.log_std_head(h), self.LOG_STD_MIN, self.LOG_STD_MAX)
        return mean, log_std

    def sample(self, x):
        mean, log_std = self.forward(x)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        x_t = dist.rsample()
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale
        log_prob = (dist.log_prob(x_t) - torch.log(1.0 - y_t.pow(2) + 1e-6)).sum(-1, keepdim=True)
        return action, log_prob, torch.tanh(mean) * self.action_scale


class TwinQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        d = state_dim + action_dim
        self.q1 = nn.Sequential(nn.Linear(d, hidden_dim), nn.ReLU(),
                                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                nn.Linear(hidden_dim, 1))
        self.q2 = nn.Sequential(nn.Linear(d, hidden_dim), nn.ReLU(),
                                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                nn.Linear(hidden_dim, 1))

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.q1(x), self.q2(x)


class DistillationDNN(nn.Module):
    """Figure 2, Step 9: maps partial_hist -> guiding actor's action."""
    def __init__(self, input_dim, action_dim, hidden_dim=256, action_scale=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Tanh(),
        )
        self.action_scale = action_scale

    def forward(self, x):
        return self.net(x) * self.action_scale

In [ ]:
# Replay Buffer (Shared D: Eq. 11)

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, full_state, partial_hist, action, reward,
             next_full_state, next_partial_hist, done, actor_id=0):
        data = (full_state, partial_hist, action, reward,
                next_full_state, next_partial_hist, float(done), actor_id)
        if len(self.buffer) < self.capacity:
            self.buffer.append(data)
        else:
            self.buffer[self.position] = data
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        idx = np.random.randint(0, len(self.buffer), size=batch_size)
        batch = [self.buffer[i] for i in idx]
        fs, ph, a, r, nfs, nph, d, aid = zip(*batch)
        return {
            "full_states":        np.array(fs, dtype=np.float32),
            "partial_hists":      np.array(ph, dtype=np.float32),
            "actions":            np.array(a, dtype=np.float32),
            "rewards":            np.array(r, dtype=np.float32).reshape(-1, 1),
            "next_full_states":   np.array(nfs, dtype=np.float32),
            "next_partial_hists": np.array(nph, dtype=np.float32),
            "dones":              np.array(d, dtype=np.float32).reshape(-1, 1),
        }

    def __len__(self):
        return len(self.buffer)

In [ ]:
# Guided SAC Agent

class GuidedSACAgent:
    def __init__(self, env_name, config=None, device=DEVICE):
        self.env_name = env_name
        self.cfg = {**DEFAULT_CONFIG, **(config or {})}
        self.device = device

        env = gym.make(env_name)
        self.state_dim = env.observation_space.shape[0]
        self.action_dim = env.action_space.shape[0]
        self.action_scale = float(env.action_space.high[0])
        env.close()

        self.obs_history = ObservationHistory(self.state_dim, self.cfg["history_len"])
        self.control_input_dim = self.obs_history.total_dim

        # Networks
        h = self.cfg["hidden_dim"]
        self.guiding_actor = GaussianPolicy(self.state_dim, self.action_dim, h, self.action_scale).to(device)
        self.control_actor = GaussianPolicy(self.control_input_dim, self.action_dim, h, self.action_scale).to(device)
        self.critic = TwinQNetwork(self.state_dim, self.action_dim, h).to(device)
        self.critic_target = deepcopy(self.critic).to(device)
        for p in self.critic_target.parameters():
            p.requires_grad = False
        self.distill_dnn = DistillationDNN(self.control_input_dim, self.action_dim, h, self.action_scale).to(device)

        # Optimizers
        self.guiding_actor_optim = optim.Adam(self.guiding_actor.parameters(), lr=self.cfg["lr"])
        self.control_actor_optim = optim.Adam(self.control_actor.parameters(), lr=self.cfg["lr"])
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=self.cfg["lr"])
        self.distill_optim = optim.Adam(self.distill_dnn.parameters(), lr=self.cfg["distill_lr"])

        # Entropy
        self.target_entropy = -self.action_dim
        self.log_alpha = torch.tensor(np.log(self.cfg["init_alpha"]),
                                      dtype=torch.float32, requires_grad=True, device=device)
        self.alpha_optim = optim.Adam([self.log_alpha], lr=self.cfg["alpha_lr"])

        self.replay_buffer = ReplayBuffer(self.cfg["replay_capacity"])

    @property
    def alpha(self):
        return self.log_alpha.exp().detach()

    def update(self):
        """Single gradient update (Algorithm 1, Steps 8-15)."""
        cfg = self.cfg
        b = self.replay_buffer.sample(cfg["batch_size"])
        s      = to_tensor(b["full_states"])
        ph     = to_tensor(b["partial_hists"])
        a      = to_tensor(b["actions"])
        r      = to_tensor(b["rewards"])
        s_next = to_tensor(b["next_full_states"])
        ph_next = to_tensor(b["next_partial_hists"])
        d      = to_tensor(b["dones"])

        # Detach alpha for actor/critic updates (avoid graph reuse)
        alpha = self.log_alpha.exp().detach()

        # Critic update (Eq. 12-13)
        with torch.no_grad():
            na, nlp, _ = self.guiding_actor.sample(s_next)
            q1t, q2t = self.critic_target(s_next, na)
            td_target = r + (1 - d) * cfg["gamma"] * (torch.min(q1t, q2t) - alpha * nlp)
        q1, q2 = self.critic(s, a)
        critic_loss = F.mse_loss(q1, td_target) + F.mse_loss(q2, td_target)
        self.critic_optim.zero_grad()
        critic_loss.backward()
        self.critic_optim.step()

        # Guiding actor update (Eq. 15)
        ag, lpg, _ = self.guiding_actor.sample(s)
        q1g, q2g = self.critic(s, ag)
        guiding_loss = (alpha * lpg - torch.min(q1g, q2g)).mean()
        self.guiding_actor_optim.zero_grad()
        guiding_loss.backward()
        self.guiding_actor_optim.step()

        # Control actor update (Eq. 14 + distillation)
        ac, lpc, _ = self.control_actor.sample(ph)
        q1c, q2c = self.critic(s, ac)
        rl_loss = (alpha * lpc - torch.min(q1c, q2c)).mean()
        with torch.no_grad():
            distill_target = self.distill_dnn(ph)
        control_loss = rl_loss + cfg["distill_coef"] * F.mse_loss(ac, distill_target)
        self.control_actor_optim.zero_grad()
        control_loss.backward()
        self.control_actor_optim.step()

        # DNN distillation update (Figure 2, Step 9)
        with torch.no_grad():
            _, _, guiding_det = self.guiding_actor.sample(s)
        distill_loss = F.mse_loss(self.distill_dnn(ph), guiding_det)
        self.distill_optim.zero_grad()
        distill_loss.backward()
        self.distill_optim.step()

        # Entropy update (Eq. 16), fresh log_alpha graph here
        # Re-sample log_prob for alpha gradient (detach everything else)
        with torch.no_grad():
            _, lpg_for_alpha, _ = self.guiding_actor.sample(s)
        alpha_loss = -(self.log_alpha.exp() * (lpg_for_alpha + self.target_entropy)).mean()
        self.alpha_optim.zero_grad()
        alpha_loss.backward()
        self.alpha_optim.step()

        # Target update (Eq. 17)
        soft_update(self.critic_target, self.critic, cfg["tau"])

        return {"critic": critic_loss.item(), "guiding": guiding_loss.item(),
                "control": control_loss.item(), "distill": distill_loss.item(),
                "alpha": alpha.item()}

    def collect_step(self, env, state, step):
        """Iterative interaction (Eq. 11): guiding/control alternate."""
        full_state = np.array(state, dtype=np.float32)
        partial_obs = get_partial_obs(full_state, self.env_name)
        is_guiding = (step % 2 == 0)

        # Get current partial history BEFORE action
        current_hist = self.obs_history.get()

        if is_guiding:
            st = to_tensor(full_state).unsqueeze(0)
            if step < self.cfg["random_start"]:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action, _, _ = self.guiding_actor.sample(st)
                action = action.cpu().numpy().flatten()
        else:
            ht = to_tensor(current_hist).unsqueeze(0)
            if step < self.cfg["random_start"]:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action, _, _ = self.control_actor.sample(ht)
                action = action.cpu().numpy().flatten()

        action = np.clip(action, -self.action_scale, self.action_scale)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        next_full = np.array(next_state, dtype=np.float32)
        next_partial = get_partial_obs(next_full, self.env_name)
        self.obs_history.push(next_partial)
        next_hist = self.obs_history.get()

        self.replay_buffer.push(
            full_state, current_hist, action, reward,
            next_full, next_hist, done, actor_id=int(not is_guiding)
        )
        return next_state, reward, done, info

    def evaluate(self, num_episodes=10, seed_offset=0):
        """Deterministic eval with control actor only."""
        env = gym.make(self.env_name)
        returns = []
        for ep in range(num_episodes):
            state, _ = env.reset(seed=seed_offset + ep)
            partial = get_partial_obs(np.array(state, np.float32), self.env_name)
            hist = ObservationHistory(self.state_dim, self.cfg["history_len"])
            hist.reset(partial)
            ret, done = 0.0, False
            while not done:
                ht = to_tensor(hist.get()).unsqueeze(0)
                with torch.no_grad():
                    _, _, det_a = self.control_actor.sample(ht)
                a = np.clip(det_a.cpu().numpy().flatten(), -self.action_scale, self.action_scale)
                ns, r, term, trunc, _ = env.step(a)
                done = term or trunc
                ret += r
                hist.push(get_partial_obs(np.array(ns, np.float32), self.env_name))
                state = ns
            returns.append(ret)
        env.close()
        return np.mean(returns), np.std(returns), returns

## 3. Baseline: SAC on partial observations

Standard SAC that receives partial observations, without guidance or history. This is the direct competitor in the paper's Table 3.

In [ ]:
class VanillaSACAgent:
    """Standard SAC on partial observations (paper baseline)."""

    def __init__(self, env_name, config=None, device=DEVICE):
        self.env_name = env_name
        self.cfg = {**DEFAULT_CONFIG, **(config or {})}
        self.device = device

        env = gym.make(env_name)
        self.state_dim = env.observation_space.shape[0]
        self.action_dim = env.action_space.shape[0]
        self.action_scale = float(env.action_space.high[0])
        env.close()

        h = self.cfg["hidden_dim"]
        # SAC operates on partial observations directly
        self.actor = GaussianPolicy(self.state_dim, self.action_dim, h, self.action_scale).to(device)
        self.critic = TwinQNetwork(self.state_dim, self.action_dim, h).to(device)
        self.critic_target = deepcopy(self.critic).to(device)
        for p in self.critic_target.parameters():
            p.requires_grad = False

        self.actor_optim = optim.Adam(self.actor.parameters(), lr=self.cfg["lr"])
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=self.cfg["lr"])

        self.target_entropy = -self.action_dim
        self.log_alpha = torch.tensor(np.log(self.cfg["init_alpha"]),
                                      dtype=torch.float32, requires_grad=True, device=device)
        self.alpha_optim = optim.Adam([self.log_alpha], lr=self.cfg["alpha_lr"])

        # Simple buffer: (partial_obs, action, reward, next_partial_obs, done)
        self.buffer = []
        self.buf_pos = 0
        self.buf_cap = self.cfg["replay_capacity"]

    @property
    def alpha(self):
        return self.log_alpha.exp().detach()

    def push(self, s, a, r, ns, d):
        data = (s, a, r, ns, float(d))
        if len(self.buffer) < self.buf_cap:
            self.buffer.append(data)
        else:
            self.buffer[self.buf_pos] = data
        self.buf_pos = (self.buf_pos + 1) % self.buf_cap

    def sample_batch(self, bs):
        idx = np.random.randint(0, len(self.buffer), size=bs)
        s, a, r, ns, d = zip(*[self.buffer[i] for i in idx])
        return (to_tensor(np.array(s)), to_tensor(np.array(a)),
                to_tensor(np.array(r).reshape(-1,1)),
                to_tensor(np.array(ns)),
                to_tensor(np.array(d).reshape(-1,1)))

    def update(self):
        s, a, r, ns, d = self.sample_batch(self.cfg["batch_size"])
        alpha = self.log_alpha.exp().detach()

        with torch.no_grad():
            na, nlp, _ = self.actor.sample(ns)
            q1t, q2t = self.critic_target(ns, na)
            td = r + (1-d) * self.cfg["gamma"] * (torch.min(q1t, q2t) - alpha * nlp)
        q1, q2 = self.critic(s, a)
        cl = F.mse_loss(q1, td) + F.mse_loss(q2, td)
        self.critic_optim.zero_grad()
        cl.backward()
        self.critic_optim.step()

        na2, lp2, _ = self.actor.sample(s)
        q1a, q2a = self.critic(s, na2)
        al = (alpha * lp2 - torch.min(q1a, q2a)).mean()
        self.actor_optim.zero_grad()
        al.backward()
        self.actor_optim.step()

        # Fresh graph for alpha update
        with torch.no_grad():
            _, lp_alpha, _ = self.actor.sample(s)
        aloss = -(self.log_alpha.exp() * (lp_alpha + self.target_entropy)).mean()
        self.alpha_optim.zero_grad()
        aloss.backward()
        self.alpha_optim.step()

        soft_update(self.critic_target, self.critic, self.cfg["tau"])
        return cl.item()

    def collect_step(self, env, state, step):
        partial = get_partial_obs(np.array(state, np.float32), self.env_name)
        if step < self.cfg["random_start"]:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action, _, _ = self.actor.sample(to_tensor(partial).unsqueeze(0))
            action = action.cpu().numpy().flatten()
        action = np.clip(action, -self.action_scale, self.action_scale)
        ns, r, term, trunc, info = env.step(action)
        done = term or trunc
        next_partial = get_partial_obs(np.array(ns, np.float32), self.env_name)
        self.push(partial, action, r, next_partial, done)
        return ns, r, done, info

    def evaluate(self, num_episodes=10, seed_offset=0):
        env = gym.make(self.env_name)
        returns = []
        for ep in range(num_episodes):
            state, _ = env.reset(seed=seed_offset + ep)
            ret, done = 0.0, False
            while not done:
                partial = get_partial_obs(np.array(state, np.float32), self.env_name)
                with torch.no_grad():
                    _, _, det_a = self.actor.sample(to_tensor(partial).unsqueeze(0))
                a = np.clip(det_a.cpu().numpy().flatten(), -self.action_scale, self.action_scale)
                ns, r, term, trunc, _ = env.step(a)
                done = term or trunc
                ret += r
                state = ns
            returns.append(ret)
        env.close()
        return np.mean(returns), np.std(returns), returns

## 4. Training function

One loop trains both the Guided SAC agent and the baseline, with a progress bar and logging.

In [ ]:
def train_agent(agent, env_name, seed=0, total_steps=None, eval_every=None,
                eval_episodes=10, verbose=True):
    """
    Unified training loop for both GuidedSACAgent and VanillaSACAgent.
    Returns dict with eval_steps, eval_means, eval_stds, wall_time.
    """
    cfg = agent.cfg
    total_steps = total_steps or cfg["total_steps"]
    eval_every = eval_every or cfg["eval_every"]

    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make(env_name)
    env.action_space.seed(seed)

    results = {"eval_steps": [], "eval_means": [], "eval_stds": [],
               "episode_returns": [], "wall_times": []}

    # Initial eval
    em, es, _ = agent.evaluate(eval_episodes, seed * 1000)
    results["eval_steps"].append(0)
    results["eval_means"].append(em)
    results["eval_stds"].append(es)

    state, _ = env.reset(seed=seed)

    # Init obs history for GuidedSAC
    if hasattr(agent, 'obs_history'):
        partial = get_partial_obs(np.array(state, np.float32), env_name)
        agent.obs_history.reset(partial)

    ep_ret = 0.0
    ep_count = 0
    t0 = time.time()

    pbar = tqdm(range(1, total_steps + 1), desc=f"{type(agent).__name__}",
                disable=not verbose, miniters=total_steps//100)

    for step in pbar:
        ns, r, done, _ = agent.collect_step(env, state, step)
        ep_ret += r

        if done:
            ep_count += 1
            results["episode_returns"].append(ep_ret)
            state, _ = env.reset(seed=seed + ep_count)
            if hasattr(agent, 'obs_history'):
                partial = get_partial_obs(np.array(state, np.float32), env_name)
                agent.obs_history.reset(partial)
            ep_ret = 0.0
        else:
            state = ns

        # Gradient updates
        buf_len = len(agent.replay_buffer) if hasattr(agent, 'replay_buffer') else len(agent.buffer)
        if step >= cfg["update_after"] and buf_len >= cfg["batch_size"]:
            if step % cfg["update_every"] == 0:
                agent.update()

        # Eval
        if step % eval_every == 0:
            em, es, _ = agent.evaluate(eval_episodes, seed * 1000)
            wt = time.time() - t0
            results["eval_steps"].append(step)
            results["eval_means"].append(em)
            results["eval_stds"].append(es)
            results["wall_times"].append(wt)
            pbar.set_postfix({"eval": f"{em:.1f}±{es:.1f}",
                              "ep": ep_count, "t": f"{wt/60:.1f}m"})

    env.close()
    results["total_wall_time"] = time.time() - t0
    return results

## 5. Validation run

A short run to check that the code works: Hopper-v4, 50K steps, one seed.

In [ ]:
# Quick test: 50K steps, single seed
QUICK_ENV = "Hopper-v4"
QUICK_STEPS = 50_000
QUICK_EVAL = 5_000

quick_cfg = {**DEFAULT_CONFIG, "total_steps": QUICK_STEPS, "eval_every": QUICK_EVAL}

print("="*60)
print("Quick Validation: Guided SAC")
print("="*60)
gsac_agent = GuidedSACAgent(QUICK_ENV, quick_cfg)
gsac_results = train_agent(gsac_agent, QUICK_ENV, seed=0,
                           total_steps=QUICK_STEPS, eval_every=QUICK_EVAL)

print("\n" + "="*60)
print("Quick Validation: Vanilla SAC")
print("="*60)
sac_agent = VanillaSACAgent(QUICK_ENV, quick_cfg)
sac_results = train_agent(sac_agent, QUICK_ENV, seed=0,
                          total_steps=QUICK_STEPS, eval_every=QUICK_EVAL)

In [ ]:
# Quick comparison plot
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(gsac_results["eval_steps"], gsac_results["eval_means"],
        label="Guided SAC", color="#2196F3", linewidth=2)
ax.plot(sac_results["eval_steps"], sac_results["eval_means"],
        label="SAC (POMDP)", color="#FF9800", linewidth=2)
ax.set_xlabel("Timesteps")
ax.set_ylabel("Average Return")
ax.set_title(f"Quick Validation: {QUICK_ENV} POMDP ({QUICK_STEPS//1000}K steps)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nGuided SAC final: {gsac_results['eval_means'][-1]:.2f}")
print(f"SAC final:         {sac_results['eval_means'][-1]:.2f}")

## 6. Full experiments

Paper Section V.B: 5 environments, Guided SAC and SAC, 10 seeds, 1M steps each; evaluation every 10K steps over 10 deterministic trajectories. A full run takes about 12 hours per environment and seed, so running everything in one Colab session is not practical; adjust the parameters below as needed.

In [ ]:
# Experiment parameters

ENVS = [
    "HalfCheetah-v4",
    "Hopper-v4",
    "InvertedDoublePendulum-v4",
    "Walker2d-v4",
    "Swimmer-v4",
]

# Paper: 10 seeds, 1M steps. Adjust for compute budget.
NUM_SEEDS = 3          # Paper: 10. Reduce for faster experiments.
TOTAL_STEPS = 200_000  # Paper: 1_000_000. Reduce for faster experiments.
EVAL_EVERY = 10_000

ALGORITHMS = {
    "Guided SAC": GuidedSACAgent,
    "SAC":        VanillaSACAgent,
}

print(f"Environments:  {len(ENVS)}")
print(f"Algorithms:    {list(ALGORITHMS.keys())}")
print(f"Seeds:         {NUM_SEEDS}")
print(f"Steps/run:     {TOTAL_STEPS:,}")
print(f"Total runs:    {len(ENVS) * len(ALGORITHMS) * NUM_SEEDS}")

In [ ]:
# Run all experiments

all_results = {}  # {env: {algo: {seed: results_dict}}}

exp_cfg = {**DEFAULT_CONFIG, "total_steps": TOTAL_STEPS, "eval_every": EVAL_EVERY}

total_runs = len(ENVS) * len(ALGORITHMS) * NUM_SEEDS
run_idx = 0

for env_name in ENVS:
    all_results[env_name] = {}
    for algo_name, AgentClass in ALGORITHMS.items():
        all_results[env_name][algo_name] = {}
        for seed in range(NUM_SEEDS):
            run_idx += 1
            print(f"\n{'='*60}")
            print(f"[{run_idx}/{total_runs}] {env_name} | {algo_name} | seed={seed}")
            print(f"{'='*60}")

            agent = AgentClass(env_name, exp_cfg)
            results = train_agent(agent, env_name, seed=seed,
                                  total_steps=TOTAL_STEPS,
                                  eval_every=EVAL_EVERY,
                                  verbose=True)
            all_results[env_name][algo_name][seed] = results

            print(f"  → Final eval: {results['eval_means'][-1]:.2f} "
                  f"(wall time: {results.get('total_wall_time', 0)/60:.1f} min)")

print(f"\n\n{'#'*60}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"{'#'*60}")

## 7. Results

In [ ]:
# Table 3 Reproduction: Maximum Average Return

print("\n" + "="*80)
print("Table 3: Maximum Average Return on POMDP MuJoCo Tasks")
print("="*80)

table_rows = []
for env_name in ENVS:
    row = {"Task": env_name.replace("-v4", "")}
    for algo_name in ALGORITHMS:
        if env_name in all_results and algo_name in all_results[env_name]:
            seed_maxes = []
            for seed, res in all_results[env_name][algo_name].items():
                seed_maxes.append(max(res["eval_means"]))
            mean_max = np.mean(seed_maxes)
            std_max = np.std(seed_maxes)
            row[algo_name] = f"{mean_max:.2f}±{std_max:.2f}"
            row[f"{algo_name}_val"] = mean_max
    # Delta
    if "Guided SAC_val" in row and "SAC_val" in row:
        best_baseline = row["SAC_val"]
        gsac_val = row["Guided SAC_val"]
        delta = (gsac_val - best_baseline) / best_baseline * 100 if best_baseline != 0 else 0
        row["Δ%"] = f"{delta:+.1f}%"
    table_rows.append(row)

df = pd.DataFrame(table_rows)
display_cols = ["Task"] + list(ALGORITHMS.keys()) + ["Δ%"]
display_cols = [c for c in display_cols if c in df.columns]
print(df[display_cols].to_string(index=False))

In [ ]:
# Learning Curves (Paper Figure 5 style)

colors = {"Guided SAC": "#2196F3", "SAC": "#FF9800"}
n_envs = len(ENVS)
fig, axes = plt.subplots(1, n_envs, figsize=(5 * n_envs, 4), squeeze=False)

for i, env_name in enumerate(ENVS):
    ax = axes[0, i]
    ax.set_title(env_name.replace("-v4", ""), fontsize=12, fontweight="bold")

    for algo_name in ALGORITHMS:
        if env_name not in all_results or algo_name not in all_results[env_name]:
            continue

        # Collect curves across seeds
        seed_curves = []
        steps = None
        for seed, res in all_results[env_name][algo_name].items():
            seed_curves.append(res["eval_means"])
            if steps is None:
                steps = res["eval_steps"]

        if not seed_curves:
            continue

        # Align lengths
        min_len = min(len(c) for c in seed_curves)
        curves = np.array([c[:min_len] for c in seed_curves])
        steps_arr = np.array(steps[:min_len])

        mean_curve = curves.mean(axis=0)
        std_curve = curves.std(axis=0)

        ax.plot(steps_arr, mean_curve, label=algo_name,
                color=colors.get(algo_name, "gray"), linewidth=2)
        ax.fill_between(steps_arr,
                        mean_curve - std_curve,
                        mean_curve + std_curve,
                        alpha=0.2, color=colors.get(algo_name, "gray"))

    ax.set_xlabel("Timesteps")
    if i == 0:
        ax.set_ylabel("Average Return")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("POMDP MuJoCo Learning Curves (mean ± std across seeds)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: learning_curves.png")

In [ ]:
# Per-Environment Detailed Plots (2-column like Paper Figure 5)

for env_name in ENVS:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(env_name.replace("-v4", ""), fontsize=14, fontweight="bold")

    # Left: Learning curve
    for algo_name in ALGORITHMS:
        if env_name not in all_results or algo_name not in all_results[env_name]:
            continue
        seed_curves = []
        steps = None
        for seed, res in all_results[env_name][algo_name].items():
            seed_curves.append(res["eval_means"])
            if steps is None:
                steps = res["eval_steps"]
        if not seed_curves:
            continue
        min_len = min(len(c) for c in seed_curves)
        curves = np.array([c[:min_len] for c in seed_curves])
        steps_arr = np.array(steps[:min_len])
        mean_c = curves.mean(0)
        std_c = curves.std(0)
        ax1.plot(steps_arr, mean_c, label=algo_name,
                 color=colors.get(algo_name, "gray"), linewidth=2)
        ax1.fill_between(steps_arr, mean_c - std_c, mean_c + std_c,
                         alpha=0.2, color=colors.get(algo_name, "gray"))

    ax1.set_xlabel("Timesteps")
    ax1.set_ylabel("Average Return")
    ax1.set_title("POMDP Learning Curve")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Right: Final performance bar chart
    algo_names = []
    final_means = []
    final_stds = []
    for algo_name in ALGORITHMS:
        if env_name not in all_results or algo_name not in all_results[env_name]:
            continue
        maxes = [max(r["eval_means"]) for r in all_results[env_name][algo_name].values()]
        algo_names.append(algo_name)
        final_means.append(np.mean(maxes))
        final_stds.append(np.std(maxes))

    bars = ax2.bar(algo_names, final_means, yerr=final_stds,
                   color=[colors.get(n, "gray") for n in algo_names],
                   capsize=5, alpha=0.8)
    ax2.set_ylabel("Max Average Return")
    ax2.set_title("Peak Performance")
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    fname = f"{env_name.replace('-v4', '')}_results.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## 8. Export

In [ ]:
# Save all results to JSON

# Convert numpy types for JSON serialization
def convert_for_json(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_for_json(v) for v in obj]
    return obj

export_data = convert_for_json(all_results)

with open("guided_sac_results.json", "w") as f:
    json.dump(export_data, f, indent=2)

print("Saved: guided_sac_results.json")
print(f"File size: {os.path.getsize('guided_sac_results.json') / 1024:.1f} KB")

In [ ]:
# Summary Table (copy-paste ready for paper)

print("\n" + "="*80)
print("SUMMARY, Paper Table 3 Format")
print("="*80)
print(f"{'Task':<30} {'SAC':>18} {'Guided SAC':>18} {'Δ':>8}")
print("-"*80)

for env_name in ENVS:
    task = env_name.replace("-v4", "")
    vals = {}
    for algo in ALGORITHMS:
        if env_name in all_results and algo in all_results[env_name]:
            maxes = [max(r["eval_means"]) for r in all_results[env_name][algo].values()]
            vals[algo] = (np.mean(maxes), np.std(maxes))

    sac_str = f"{vals['SAC'][0]:.1f}±{vals['SAC'][1]:.1f}" if "SAC" in vals else "-"
    gsac_str = f"{vals['Guided SAC'][0]:.1f}±{vals['Guided SAC'][1]:.1f}" if "Guided SAC" in vals else "-"

    if "SAC" in vals and "Guided SAC" in vals:
        best_base = vals["SAC"][0]
        delta = (vals["Guided SAC"][0] - best_base) / best_base * 100 if best_base != 0 else 0
        delta_str = f"{delta:+.0f}%"
    else:
        delta_str = "-"

    print(f"{task:<30} {sac_str:>18} {gsac_str:>18} {delta_str:>8}")

print("="*80)

## Notes

| Parameter | Paper | Notebook default | Full run |
|---|---|---|---|
| Total steps | 1M | 200K | 1M |
| Seeds | 10 | 3 | 10 |
| Evaluation frequency | 10K | 10K | 10K |
| Evaluation episodes | 10 | 10 | 10 |

For the full replication set `NUM_SEEDS = 10` and `TOTAL_STEPS = 1_000_000` in Section 6.

Compute estimates: validation run (50K steps) about 5 minutes; one environment, 3 seeds, 200K steps about 1 to 2 hours; the full protocol (5 environments, 2 algorithms, 10 seeds, 1M steps) about 120 hours.